# ResonanceOS v6 AI System – Live Evolution & Reinforcement Tracking
## Developer Notebook: HR-PPO Evolution Over Iterations

This notebook demonstrates:
- Multi-tenant, multi-profile HRV evolution
- Live HR-PPO reinforcement learning over multiple iterations
- Metrics collection (HRV, sentiment, emotion)
- System-wide visualizations of resonance evolution
- Aurora Core integration with dynamic HRV updates

In [ ]:
# Import modules
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from resonance_os.generation.human_resonant_writer import HumanResonantWriter
from resonance_os.profiles.multi_tenant_hr_profiles import HRVProfileManager
from resonance_os.evolution.hr_rl_trainer import HRWritingEnv, train_hr_ppo
from resonance_os.integrations.aurora_hr_adapter import AuroraHRAdapter
from resonance_os.utils.text_emotion_tools import sentiment_score, emotion_intensity

print('Modules loaded successfully')

## 1️⃣ Setup Multi-Tenant Profiles & Writer

In [ ]:
# Load profiles
profile_manager = HRVProfileManager('./profiles/hr_profiles')
tenants_profiles = [
    ('default', 'brand_identity_v1'),
    ('tech', 'tech_innovation_v1')
]

hrv_profiles = {tp: profile_manager.load_profile(*tp) for tp in tenants_profiles}

writer = HumanResonantWriter()
aurora_adapter = AuroraHRAdapter()

print('Profiles, Writer, and Aurora Adapter initialized')

## 2️⃣ Initialize System Metrics Containers

In [ ]:
iterations = 5  # simulate 5 reinforcement iterations
system_evolution = {}

for tp in tenants_profiles:
    system_evolution[tp] = {
        'hrv_feedback_history': [],
        'sentiment_history': [],
        'emotion_history': []
    }

print('System metrics containers initialized')

## 3️⃣ Simulate Multi-Iteration HR-PPO Reinforcement

In [ ]:
for iteration in range(iterations):
    print(f'--- Iteration {iteration+1} ---')
    for tp in tenants_profiles:
        prompt = f'Generate human-resonant article for tenant {tp[0]}, profile {tp[1]}'
        article = writer.generate(prompt)
        paragraphs = [p.strip() for p in article.split('.') if p.strip()]
        
        # Simulated HRF feedback
        hrv_feedback = np.random.rand(len(paragraphs), len(hrv_profiles[tp]))
        sentiments = [sentiment_score(p) for p in paragraphs]
        emotions = [emotion_intensity(p) for p in paragraphs]
        
        # Store metrics
        system_evolution[tp]['hrv_feedback_history'].append(hrv_feedback)
        system_evolution[tp]['sentiment_history'].append(sentiments)
        system_evolution[tp]['emotion_history'].append(emotions)
        
        print(f'Tenant {tp[0]}, profile {tp[1]}: {len(paragraphs)} paragraphs, HRV shape {hrv_feedback.shape}')

## 4️⃣ Train HR-PPO on Latest Iteration

In [ ]:
env = HRWritingEnv(hrv_dim=len(hrv_profiles[tenants_profiles[0]]))
ppo_model = train_hr_ppo(env, timesteps=2000)
print('✅ HR-PPO trained on latest iteration for resonance optimization')

## 5️⃣ Visualize HRV Evolution Across Iterations

In [ ]:
plt.figure(figsize=(14,7))
for tp in tenants_profiles:
    for dim in range(len(hrv_profiles[tp])):
        # Average across paragraphs for each iteration
        evolution_curve = [np.mean(it[:, dim]) for it in system_evolution[tp]['hrv_feedback_history']]
        plt.plot(evolution_curve, label=f'{tp} - HRV Dim {dim}')

plt.title('HRV Evolution Across Iterations')
plt.xlabel('Iteration')
plt.ylabel('Mean HRV Score')
plt.legend()
plt.show()

## 6️⃣ Aurora Core Feedback Update
Inject latest mean HRV vector from first tenant-profile as dynamic feedback into Aurora Core.

In [ ]:
latest_hrv_mean = np.mean(system_evolution[tenants_profiles[0]]['hrv_feedback_history'][-1], axis=0)
aurora_adapter.inject_hrv_feedback(latest_hrv_mean)
print(f'Injected latest HRV feedback into Aurora Core for {tenants_profiles[0]}')

## 7️⃣ Summary & Developer Notes
- Multi-iteration reinforcement learning simulated
- HRV, sentiment, and emotion tracked across iterations
- System evolution visualized per HRV dimension
- Aurora Core receives dynamic HRV updates

**Next Steps:**
1. Replace simulated HRF with ML-based prediction
2. Expand PPO training for multi-tenant joint optimization
3. Persist HRV evolution to database for analytics and version tracking